# Gold Layer – disease_medication_pharmacy_gold

## Purpose
This notebook creates the main Gold analytical table that connects diseases, medications, pharmacies, and inventory availability into a single business-ready dataset.

## Business Goal
The purpose of this table is to answer the core business question:

**For a given disease, where in the United States can patients find the required medications?**

## Source Tables
This Gold table is built from the following Silver layer tables:
- `capstone.silver.diseases`
- `capstone.silver.medications`
- `capstone.silver.inventory`
- `capstone.silver.pharmacy`

## Output Table
- `capstone.gold.disease_medication_pharmacy_gold`

## Key Business Capabilities Supported
This table supports:
- disease → medication → pharmacy relationship mapping
- medication availability analysis by city, state, and region
- pharmacy-level reporting
- price and quantity analysis
- downstream dashboard reporting

## Data Quality Rules Applied
The transformation includes the following quality checks:
- exclude null disease, medication, and pharmacy identifiers
- exclude inventory rows with null or non-positive quantity
- exclude invalid negative pricing values
- remove duplicate source rows using `DISTINCT`

## Grain
One row in this table represents:

**one disease + one medication + one pharmacy combination with its available quantity and price**

## Notes
This table is designed as the primary Gold fact-style dataset for downstream analytical aggregations and dashboard visualizations.

In [0]:
%sql
-- ============================================================
-- Gold Table: disease_medication_pharmacy_gold
-- Purpose:
-- Create the main analytical Gold dataset that links diseases,
-- medications, pharmacies, and inventory availability.
--
-- Business question supported:
-- For a given disease, where can the required medications be
-- found across pharmacies and geographic regions in the U.S.?
--
-- Target table:
-- capstone.gold.disease_medication_pharmacy_gold
-- ============================================================

CREATE OR REPLACE TABLE capstone.gold.disease_medication_pharmacy_gold
USING DELTA
AS

WITH

-- ------------------------------------------------------------
-- Step 1: Select valid disease reference data
-- Only keep business-critical disease identifiers and names
-- ------------------------------------------------------------
disease_base AS (
    SELECT DISTINCT
        disease_id,
        disease_name
    FROM capstone.silver.diseases
    WHERE disease_id IS NOT NULL
      AND disease_name IS NOT NULL
),

-- ------------------------------------------------------------
-- Step 2: Select valid medication reference data
-- Keep medication records that are properly linked to diseases
-- ------------------------------------------------------------
medication_base AS (
    SELECT DISTINCT
        medication_id,
        medication_name,
        disease_id
    FROM capstone.silver.medications
    WHERE medication_id IS NOT NULL
      AND medication_name IS NOT NULL
      AND disease_id IS NOT NULL
),

-- ------------------------------------------------------------
-- Step 3: Select valid pharmacy reference data
-- Keep pharmacy and geographic attributes required for analysis
-- ------------------------------------------------------------
pharmacy_base AS (
    SELECT DISTINCT
        pharmacy_id,
        pharmacy_name,
        city,
        state,
        region
    FROM capstone.silver.pharmacy
    WHERE pharmacy_id IS NOT NULL
      AND pharmacy_name IS NOT NULL
),

-- ------------------------------------------------------------
-- Step 4: Clean inventory data
-- Apply quality rules for quantity and price
-- ------------------------------------------------------------
inventory_base AS (
    SELECT
        pharmacy_id,
        medication_id,
        price,
        quantity
    FROM capstone.silver.inventory
    WHERE pharmacy_id IS NOT NULL
      AND medication_id IS NOT NULL
      AND quantity IS NOT NULL
      AND quantity > 0
      AND (price IS NULL OR price >= 0)
),

-- ------------------------------------------------------------
-- Step 5: Remove duplicate inventory rows
-- DISTINCT is used as a safeguard against repeated records
-- ------------------------------------------------------------
inventory_deduplicated AS (
    SELECT DISTINCT
        pharmacy_id,
        medication_id,
        price,
        quantity
    FROM inventory_base
),

-- ------------------------------------------------------------
-- Step 6: Build the final Gold dataset
-- Join diseases, medications, inventory, and pharmacies
-- ------------------------------------------------------------
gold_dataset AS (
    SELECT
        d.disease_id,
        d.disease_name,
        m.medication_id,
        m.medication_name,
        p.pharmacy_id,
        p.pharmacy_name,
        p.city,
        p.state,
        p.region,
        i.price,
        i.quantity
    FROM disease_base d
    JOIN medication_base m
        ON d.disease_id = m.disease_id
    JOIN inventory_deduplicated i
        ON m.medication_id = i.medication_id
    JOIN pharmacy_base p
        ON i.pharmacy_id = p.pharmacy_id
)

-- ------------------------------------------------------------
-- Final output
-- DISTINCT is kept as a final protection against duplicate rows
-- ------------------------------------------------------------
SELECT DISTINCT *
FROM gold_dataset;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT disease_id) AS distinct_diseases,
    COUNT(DISTINCT medication_id) AS distinct_medications,
    COUNT(DISTINCT pharmacy_id) AS distinct_pharmacies
FROM capstone.gold.disease_medication_pharmacy_gold;

In [0]:
%sql
SELECT *
FROM capstone.gold.disease_medication_pharmacy_gold
LIMIT 20;